# Threat-Aware Adaptive Eavesdropping in QKD

This notebook begins the revised study: **maximize unmet application key demand under an attack budget and a calibrated detection-risk constraint**. The simulator separates the demand process from its metadata channel: `metadata_coupling=0` is a null control, while demand and key consumption remain unchanged.

The first research gate is model-based: calibrate the detector, measure metadata predictive value, then compare constrained no-metadata and HMM-belief attackers. PPO follows only if this value-of-information experiment supports the hypothesis. The earlier value-targeted prototype is preserved in `main_value_targeted_legacy.ipynb`.

In [ ]:
# %% Stage 1 ? imports
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from threat_aware_qkd import (
    SimConfig, AvailabilityEnv, BeliefPolicy, DetectorAwarePolicy,
    PeriodicPolicy, RandomPolicy, StateOraclePolicy,
    calibrate_cusum_threshold, evaluate_policy, metadata_predictive_logloss,
    select_under_detection_constraint,
)

SEED_TUNE = 10_000
SEED_TEST = 100_000


## Controlled operating point

Load is defined as expected application demand divided by attack-free key supply. Pool capacity, demand-state persistence, metadata coupling, finite-sample QBER, CUSUM false-alarm target, and Eve's cumulative `sum(f_t)` budget are explicit parameters. Demand is Poisson conditional on a three-state HMM. Metadata is generated from the real state with probability `rho`, and otherwise from an independent state drawn from the stationary prior.

In [ ]:
# %% Stage 2 ? calibrate the attack-free detector
base = SimConfig(
    horizon=240, qber_baseline=0.01, qber_sample_size=500,
    cusum_reference=0.003, false_alarm_target=0.01,
    raw_key_bits_per_round=1000.0, pool_capacity=5000.0,
    initial_pool_fraction=1.0, load=1.0, metadata_coupling=0.5,
    persistence=0.90, attack_budget=3.0, observe_pool=False,
)
base.cusum_threshold = calibrate_cusum_threshold(
    base.qber_baseline, base.qber_sample_size, base.horizon,
    base.cusum_reference, base.false_alarm_target, episodes=6000, seed=9127,
)
print(f"Calibrated CUSUM h={base.cusum_threshold:.5f} for target false alarm {base.false_alarm_target:.1%}")
print("Attack-free supply:", round(base.raw_key_bits_per_round * __import__('threat_aware_qkd').bb84_key_fraction(base.qber_baseline), 1))
print("Mean demand:", round(float(base.prior @ base.demand_rates), 1), "load:", base.load)


In [ ]:
# %% Attack-free detector check
no_attack = lambda seed: (lambda obs, t: 0.0)
false_alarm_df = evaluate_policy(
    base, no_attack, episodes=500, seed0=SEED_TEST, metadata_visible=False
)
print(f"Empirical attack-free detection rate: {false_alarm_df.detected.mean():.3%}")


## Is metadata predictive?

Use held-out log-loss for the hidden application state. The no-metadata predictor uses the HMM transition prior; the metadata predictor applies the exact forward emission update. `rho=0` should produce approximately zero predictive gain. This measures prediction, while the constrained policy experiment below measures whether prediction has operational value.

In [ ]:
# %% Predictive log-loss by coupling
rho_rows = []
for rho in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
    cfg = replace(base, metadata_coupling=rho)
    row = metadata_predictive_logloss(cfg, episodes=150, seed0=200_000)
    rho_rows.append(dict(rho=rho, **row))
predictability_df = pd.DataFrame(rho_rows)
display(predictability_df)
ax = predictability_df.plot(x="rho", y="delta_logloss", marker="o", legend=False, figsize=(6, 4))
ax.axhline(0, color="black", lw=0.8); ax.set_ylabel("log-loss reduction (nats/state)")
ax.set_title("Predictive value of application metadata"); ax.grid(alpha=0.25)
plt.tight_layout()


## Constrained policy comparison

Tune the burst-belief threshold and CUSUM throttle on validation episodes only. Select the policy with the highest unmet demand subject to `P(detected) <= delta`, then report on disjoint test seeds. The no-metadata HMM is the best tuned stationary-belief reference; the metadata HMM filters the traffic observations. A pool-aware HMM is reported separately because pool visibility is a distinct information channel. The state oracle is an upper bound, not an attacker in the threat model.

In [ ]:
# %% Validation tuning and held-out evaluation
DELTA = 0.05
VALIDATION_EPISODES = 80
TEST_EPISODES = 200
THRESHOLDS = (0.10, 0.20, 0.30, 0.45, 0.60, 0.80)

def tune_belief_policy(cfg, use_metadata, pool_aware=False):
    return select_under_detection_constraint(
        cfg,
        lambda state_th, cusum_th: BeliefPolicy(
            cfg, use_metadata=use_metadata, pool_aware=pool_aware,
            state_threshold=state_th, cusum_limit=cusum_th,
        ),
        detection_limit=DELTA, threshold_grid=THRESHOLDS,
        episodes=VALIDATION_EPISODES, seed0=SEED_TUNE,
        metadata_visible=use_metadata, pool_visible=pool_aware,
    )

selected = {}
for name, use_metadata, pool_aware in [
    ("no_metadata", False, False),
    ("metadata_hmm", True, False),
    ("metadata_plus_pool", True, True),
]:
    selection, _ = tune_belief_policy(base, use_metadata, pool_aware)
    selected[name] = selection
    print(name, selection)

def tuned_factory(name, cfg=base):
    row = selected[name]
    return lambda seed: BeliefPolicy(
        cfg, use_metadata=name != "no_metadata",
        pool_aware=name == "metadata_plus_pool",
        state_threshold=row["state_threshold"], cusum_limit=row["cusum_limit"],
    )

# No-attack reference uses the same test seeds as every attacker.
no_attack_test = evaluate_policy(base, no_attack, TEST_EPISODES, SEED_TEST, metadata_visible=False)
comparison = []
for name in ["no_metadata", "metadata_hmm", "metadata_plus_pool"]:
    df = evaluate_policy(
        base, tuned_factory(name), TEST_EPISODES, SEED_TEST,
        metadata_visible=name != "no_metadata", pool_visible=name == "metadata_plus_pool",
    )
    df["incremental_damage"] = df.damage.to_numpy() - no_attack_test.damage.to_numpy()
    comparison.append(df.assign(policy=name))

# Strong reference ladder on the same independent test seed range.
for name, factory, metadata, pool in [
    ("random", lambda i: RandomPolicy(base, i), True, False),
    ("periodic", lambda i: PeriodicPolicy(base, period=8), True, False),
    ("detector_aware", lambda i: DetectorAwarePolicy(base, cusum_limit=0.5), False, False),
    ("state_oracle", lambda i: StateOraclePolicy(base, cusum_limit=0.5), False, False),
]:
    df = evaluate_policy(base, factory, TEST_EPISODES, SEED_TEST, metadata, pool)
    df["incremental_damage"] = df.damage.to_numpy() - no_attack_test.damage.to_numpy()
    comparison.append(df.assign(policy=name))

comparison_df = pd.concat(comparison, ignore_index=True)
summary_df = comparison_df.groupby("policy").agg(
    damage=("damage", "mean"), incremental_damage=("incremental_damage", "mean"),
    detection_rate=("detected", "mean"), budget_used=("attack_budget_used", "mean"),
    damage_rounds=("damage_rounds", "mean"), pool_fraction=("mean_pool_fraction", "mean"),
).sort_values("incremental_damage", ascending=False)
display(summary_df)


In [ ]:
# %% Damage versus detection risk
fig, ax = plt.subplots(figsize=(8, 5))
for name, group in comparison_df.groupby("policy"):
    ax.scatter(group.detected.astype(float).mean(), group.incremental_damage.mean(), s=65, label=name)
ax.axvline(DELTA, color="red", ls="--", label=f"risk limit {DELTA:.0%}")
ax.set_xlabel("detection probability"); ax.set_ylabel("incremental unmet demand")
ax.set_title("Availability damage under stealth risk"); ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.tight_layout()


## Value of information and operating-regime map

For a fixed operating point, value of metadata is the test-set damage difference between the tuned metadata policy and the tuned no-metadata policy, with both selected under the same detection constraint and evaluated on paired seeds. The matrix helper varies load, pool capacity, coupling, and persistence. Use it as the early kill criterion before committing to PPO.

In [ ]:
# %% Paired value of information at the current operating point
voi = float(
    comparison_df.loc[comparison_df.policy == "metadata_hmm", "incremental_damage"].mean()
    - comparison_df.loc[comparison_df.policy == "no_metadata", "incremental_damage"].mean()
)
print(f"Test-set VOI (unmet-demand units): {voi:.2f}")


In [ ]:
# %% Regime-map runner ? call with reduced grids first; this is intentionally not auto-run
def run_voi_matrix(base_cfg, loads=(0.85, 1.0, 1.10),
                   pool_sizes=(2000.0, 5000.0),
                   rhos=(0.0, 0.25, 0.5, 0.75, 1.0),
                   persistences=(0.70, 0.90), delta=DELTA,
                   validation_episodes=50, test_episodes=200):
    rows = []
    for load in loads:
        for pool_size in pool_sizes:
            for rho in rhos:
                for persistence in persistences:
                    cfg = replace(base_cfg, load=load, pool_capacity=pool_size,
                                  metadata_coupling=rho, persistence=persistence)
                    tuned = {}
                    for name, use_meta in [("no_metadata", False), ("metadata", True)]:
                        selection, _ = select_under_detection_constraint(
                            cfg, lambda st, cl, um=use_meta: BeliefPolicy(
                                cfg, use_metadata=um, state_threshold=st, cusum_limit=cl),
                            detection_limit=delta, threshold_grid=THRESHOLDS,
                            episodes=validation_episodes, seed0=SEED_TUNE,
                            metadata_visible=use_meta, pool_visible=False)
                        factory = lambda i, row=selection, um=use_meta: BeliefPolicy(
                            cfg, use_metadata=um, state_threshold=row["state_threshold"],
                            cusum_limit=row["cusum_limit"])
                        test = evaluate_policy(cfg, factory, test_episodes, SEED_TEST, use_meta, False)
                        tuned[name] = (selection, test)
                    j0 = tuned["no_metadata"][1].damage.to_numpy()
                    j1 = tuned["metadata"][1].damage.to_numpy()
                    rows.append(dict(load=load, pool_capacity=pool_size, rho=rho,
                        persistence=persistence, damage_no_metadata=float(j0.mean()),
                        damage_metadata=float(j1.mean()), voi=float((j1-j0).mean()),
                        detection_no_metadata=float(tuned["no_metadata"][1].detected.mean()),
                        detection_metadata=float(tuned["metadata"][1].detected.mean())))
    return pd.DataFrame(rows)

# Example (computationally substantial):
# voi_grid_df = run_voi_matrix(base, loads=(0.9, 1.0, 1.1), pool_sizes=(2000, 5000),
#                              rhos=(0, 0.5, 1), persistences=(0.7, 0.9))


## Next stage

Proceed to constrained PPO and recurrent PPO only after the model-based HMM/heuristic value-of-information results establish a nontrivial regime. Then compare them against the same risk-constrained validation procedure and held-out test seeds. The simulator module is `threat_aware_qkd.py`; all episode-level metrics are unmet demand, detection, attack-budget use, pool level, and overflow.